# Assignment Beta, Question 4
Hafsa Rashid, 23K-0064

Evaluating an LLM sentiment classifier (Positive, Negative, Neutral) on 300 reviews, before and after prompt changes, plus BLEU, ROUGE, METEOR and perplexity for one generated justification.


In [1]:
import math
import numpy as np
import pandas as pd

labels = ["Positive", "Negative", "Neutral"]

# rows = actual, columns = predicted
cm1 = np.array([[100,  5, 15],
                [  8, 80, 12],
                [ 20, 10, 50]])

cm2 = np.array([[106,  4, 10],
                [  6, 84, 10],
                [  9,  7, 64]])

print("Stage 1 total:", cm1.sum(), "| Stage 2 total:", cm2.sum())

Stage 1 total: 300 | Stage 2 total: 300


## 1. Accuracy with a 95% confidence interval (zero-shot)

In [2]:
def accuracy_ci(cm, z=1.96):
    n = cm.sum()
    acc = np.trace(cm) / n
    # normal approximation (Wald)
    se = math.sqrt(acc * (1 - acc) / n)
    wald = (acc - z * se, acc + z * se)
    # Wilson interval, which behaves better near 0 or 1
    centre = (acc + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    wilson = (centre - half, centre + half)
    return acc, se, wald, wilson

acc1, se1, wald1, wil1 = accuracy_ci(cm1)
print(f"correct = {np.trace(cm1)} of {cm1.sum()}")
print(f"accuracy = {acc1:.4f}")
print(f"SE = sqrt({acc1:.4f} x {1-acc1:.4f} / 300) = {se1:.4f}")
print(f"95% CI (Wald)   = [{wald1[0]:.4f}, {wald1[1]:.4f}]")
print(f"95% CI (Wilson) = [{wil1[0]:.4f}, {wil1[1]:.4f}]")

correct = 230 of 300
accuracy = 0.7667
SE = sqrt(0.7667 x 0.2333 / 300) = 0.0244
95% CI (Wald)   = [0.7188, 0.8145]
95% CI (Wilson) = [0.7156, 0.8110]


## 2. Precision, recall and F1 per class (zero-shot)

In [3]:
def per_class(cm):
    tp = np.diag(cm)
    precision = tp / cm.sum(axis=0)     # column totals = predicted
    recall = tp / cm.sum(axis=1)        # row totals = actual
    f1 = 2 * precision * recall / (precision + recall)
    support = cm.sum(axis=1)
    return pd.DataFrame({"TP": tp, "predicted": cm.sum(axis=0), "actual": support,
                         "precision": precision, "recall": recall, "F1": f1}, index=labels)

pc1 = per_class(cm1)
pc1.round(4)

,TP,predicted,actual,precision,recall,F1
Positive,100,128,120,0.7812,0.8333,0.8065
Negative,80,95,100,0.8421,0.8000,0.8205
Neutral,50,77,80,0.6494,0.6250,0.6369


## 3. Macro, weighted and micro F1 (zero-shot)

In [4]:
def averaged_f1(cm):
    pc = per_class(cm)
    macro = pc.F1.mean()
    weighted = (pc.F1 * pc.actual).sum() / pc.actual.sum()
    # micro: pool TP, FP and FN over classes
    tp = np.diag(cm).sum()
    fp = cm.sum() - tp          # every wrong prediction is one FP for some class
    fn = cm.sum() - tp          # and one FN for another
    micro_p = tp / (tp + fp)
    micro_r = tp / (tp + fn)
    micro = 2 * micro_p * micro_r / (micro_p + micro_r)
    return macro, weighted, micro

m1 = averaged_f1(cm1)
print(f"macro F1    = {m1[0]:.4f}")
print(f"weighted F1 = {m1[1]:.4f}")
print(f"micro F1    = {m1[2]:.4f}  (same as accuracy for single label multi-class)")

macro F1    = 0.7546
weighted F1 = 0.7659
micro F1    = 0.7667  (same as accuracy for single label multi-class)


## 4. What goes wrong in Stage 1

Looking at the errors directly from the confusion matrix:

In [5]:
errors = []
for i, a in enumerate(labels):
    for j, p in enumerate(labels):
        if i != j:
            errors.append((a, p, cm1[i, j]))
pd.DataFrame(errors, columns=["actual", "predicted", "count"]).sort_values("count", ascending=False)

,actual,predicted,count
4,Neutral,Positive,20
1,Positive,Neutral,15
3,Negative,Neutral,12
5,Neutral,Negative,10
2,Negative,Positive,8
0,Positive,Negative,5


Neutral is the weak class: recall is only 62.5%, and 20 neutral reviews are pushed to Positive. Positive also loses 15 reviews to Neutral, and Neutral has the lowest precision. So the model has no clear idea of where "Neutral" starts and ends, especially for mixed reviews. Negative is the most reliable class.

## 5. Stage 2 (improved prompt) with the same metrics

In [6]:
acc2, se2, wald2, wil2 = accuracy_ci(cm2)
pc2 = per_class(cm2)
m2 = averaged_f1(cm2)

print(f"accuracy = {np.trace(cm2)}/300 = {acc2:.4f}")
print(f"95% CI (Wald)   = [{wald2[0]:.4f}, {wald2[1]:.4f}]")
print(f"95% CI (Wilson) = [{wil2[0]:.4f}, {wil2[1]:.4f}]")
print(f"macro F1 = {m2[0]:.4f} | weighted F1 = {m2[1]:.4f} | micro F1 = {m2[2]:.4f}")
pc2.round(4)

accuracy = 254/300 = 0.8467
95% CI (Wald)   = [0.8059, 0.8874]
95% CI (Wilson) = [0.8015, 0.8830]
macro F1 = 0.8406 | weighted F1 = 0.8472 | micro F1 = 0.8467


,TP,predicted,actual,precision,recall,F1
Positive,106,121,120,0.8760,0.8833,0.8797
Negative,84,95,100,0.8842,0.8400,0.8615
Neutral,64,84,80,0.7619,0.8000,0.7805


In [7]:
compare = pd.DataFrame({
    "Stage 1": [acc1, m1[0], m1[1], m1[2]] + list(pc1.F1),
    "Stage 2": [acc2, m2[0], m2[1], m2[2]] + list(pc2.F1),
}, index=["accuracy", "macro F1", "weighted F1", "micro F1"] + [f"F1 {l}" for l in labels])
compare["change"] = compare["Stage 2"] - compare["Stage 1"]
compare.round(4)

,Stage 1,Stage 2,change
accuracy,0.7667,0.8467,0.0800
macro F1,0.7546,0.8406,0.0859
weighted F1,0.7659,0.8472,0.0812
micro F1,0.7667,0.8467,0.0800
F1 Positive,0.8065,0.8797,0.0732
F1 Negative,0.8205,0.8615,0.0410
F1 Neutral,0.6369,0.7805,0.1435


A rough check that the gain is not just noise: the two Wald intervals barely overlap. A proper paired test (McNemar) would need the per-review predictions, which we don't have, so I only note it.

## 6. Prompt stability over 5 example orderings

In [8]:
runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
mean = runs.mean()
std_sample = runs.std(ddof=1)     # sample std, since these 5 runs are a sample of possible orderings
std_pop = runs.std(ddof=0)
print(f"mean = {mean:.4f}")
print(f"sample std (n-1) = {std_sample:.4f}")
print(f"population std (n) = {std_pop:.4f}")
print(f"range = {runs.min():.3f} to {runs.max():.3f} ({(runs.max()-runs.min())*100:.1f} points)")
print(f"coefficient of variation = {std_sample/mean*100:.2f}%")

mean = 0.8454
sample std (n-1) = 0.0067
population std (n) = 0.0060
range = 0.837 to 0.853 (1.6 points)
coefficient of variation = 0.80%


## 7. BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L and METEOR

In [9]:
ref = "the delivery was late but the product quality is good".split()
cand = "the product quality is good but delivery was slow".split()
print(len(ref), "reference tokens:", ref)
print(len(cand), "candidate tokens:", cand)

from collections import Counter

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

10 reference tokens: ['the', 'delivery', 'was', 'late', 'but', 'the', 'product', 'quality', 'is', 'good']
9 candidate tokens: ['the', 'product', 'quality', 'is', 'good', 'but', 'delivery', 'was', 'slow']


### BLEU-4 (one reference, uniform weights 0.25, no smoothing)

In [10]:
precisions = []
for n in range(1, 5):
    c = Counter(ngrams(cand, n))
    r = Counter(ngrams(ref, n))
    clipped = sum(min(cnt, r[g]) for g, cnt in c.items())
    total = sum(c.values())
    precisions.append(clipped / total)
    print(f"p{n} = {clipped}/{total} = {clipped/total:.4f}")

c_len, r_len = len(cand), len(ref)
bp = 1.0 if c_len > r_len else math.exp(1 - r_len / c_len)
print(f"brevity penalty = exp(1 - {r_len}/{c_len}) = {bp:.4f}")

bleu4 = bp * math.exp(sum(0.25 * math.log(p) for p in precisions))
print(f"BLEU-4 = {bleu4:.4f}")

p1 = 8/9 = 0.8889
p2 = 5/8 = 0.6250
p3 = 3/7 = 0.4286
p4 = 2/6 = 0.3333
brevity penalty = exp(1 - 10/9) = 0.8948
BLEU-4 = 0.4750


### ROUGE-1, ROUGE-2 and ROUGE-L (F-measure)

In [11]:
def rouge_n(ref, cand, n):
    r = Counter(ngrams(ref, n))
    c = Counter(ngrams(cand, n))
    overlap = sum((r & c).values())
    rec = overlap / sum(r.values())
    prec = overlap / sum(c.values())
    f = 0 if overlap == 0 else 2 * prec * rec / (prec + rec)
    return overlap, prec, rec, f

def lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

rouge = {}
for n in (1, 2):
    ov, p, r, f = rouge_n(ref, cand, n)
    rouge[f"ROUGE-{n}"] = (p, r, f)
    print(f"ROUGE-{n}: overlap {ov}, P = {p:.4f}, R = {r:.4f}, F1 = {f:.4f}")

lcs = lcs_len(ref, cand)
p_l, r_l = lcs / len(cand), lcs / len(ref)
f_l = 2 * p_l * r_l / (p_l + r_l)
rouge["ROUGE-L"] = (p_l, r_l, f_l)
print(f"ROUGE-L: LCS = {lcs}, P = {p_l:.4f}, R = {r_l:.4f}, F1 = {f_l:.4f}")

ROUGE-1: overlap 8, P = 0.8889, R = 0.8000, F1 = 0.8421
ROUGE-2: overlap 5, P = 0.6250, R = 0.5556, F1 = 0.5882
ROUGE-L: LCS = 5, P = 0.5556, R = 0.5000, F1 = 0.5263


### METEOR (exact matching only)

Standard parameters (alpha = 0.9, beta = 3, gamma = 0.5, same as nltk). I only use exact word matches, because the stem and synonym modules need WordNet. "late" and "slow" are not synonyms in WordNet anyway, so this should not change the result.

In [12]:
# METEOR picks the alignment with the most matches and, among those, the fewest chunks.
# "the" appears twice in the reference, so I try every option and keep the best one.
from itertools import product

options = [[j for j, rw in enumerate(ref) if rw == w] + [None] for w in cand]

def count_chunks(al):
    if not al:
        return 0
    ch = 1
    for (i1, j1), (i2, j2) in zip(al, al[1:]):
        if not (i2 == i1 + 1 and j2 == j1 + 1):
            ch += 1
    return ch

best = None
for choice in product(*options):
    js = [j for j in choice if j is not None]
    if len(js) != len(set(js)):          # a reference word can only be used once
        continue
    al = [(i, j) for i, j in enumerate(choice) if j is not None]
    key = (len(al), -count_chunks(al))
    if best is None or key > best[0]:
        best = (key, al)
alignment = best[1]

m = len(alignment)
P = m / len(cand)
R = m / len(ref)
fmean = P * R / (0.9 * P + 0.1 * R)

# chunks: runs of matches that are next to each other in both sentences
chunks = count_chunks(alignment)

penalty = 0.5 * (chunks / m) ** 3
meteor = fmean * (1 - penalty)
print("alignment (cand index, ref index):", alignment)
print(f"matches m = {m}, P = {P:.4f}, R = {R:.4f}")
print(f"Fmean = P*R / (0.9P + 0.1R) = {fmean:.4f}")
print(f"chunks = {chunks}, penalty = 0.5 x ({chunks}/{m})^3 = {penalty:.4f}")
print(f"METEOR = {meteor:.4f}")

alignment (cand index, ref index): [(0, 5), (1, 6), (2, 7), (3, 8), (4, 9), (5, 4), (6, 1), (7, 2)]
matches m = 8, P = 0.8889, R = 0.8000
Fmean = P*R / (0.9P + 0.1R) = 0.8081
chunks = 3, penalty = 0.5 x (3/8)^3 = 0.0264
METEOR = 0.7868


## 8. Perplexity of the generated justification

In [13]:
probs = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
for tok, p in zip(cand, probs):
    print(f"{tok:<8} p = {p:.2f}  -ln p = {-math.log(p):.4f}")

nll = -np.log(probs).sum()
avg_nll = nll / len(probs)
ppl = math.exp(avg_nll)
print(f"\ntotal negative log-likelihood = {nll:.4f}")
print(f"average = {nll:.4f} / 9 = {avg_nll:.4f}")
print(f"perplexity = exp({avg_nll:.4f}) = {ppl:.4f}")
print(f"geometric mean token probability = {1/ppl:.4f}")

the      p = 0.60  -ln p = 0.5108
product  p = 0.40  -ln p = 0.9163
quality  p = 0.50  -ln p = 0.6931
is       p = 0.80  -ln p = 0.2231
good     p = 0.70  -ln p = 0.3567
but      p = 0.30  -ln p = 1.2040
delivery p = 0.25  -ln p = 1.3863
was      p = 0.50  -ln p = 0.6931
slow     p = 0.20  -ln p = 1.6094

total negative log-likelihood = 7.5929
average = 7.5929 / 9 = 0.8437
perplexity = exp(0.8437) = 2.3249
geometric mean token probability = 0.4301


## Cross check with libraries

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def expand(cm):
    y_true, y_pred = [], []
    for i in range(3):
        for j in range(3):
            y_true += [i] * cm[i, j]
            y_pred += [j] * cm[i, j]
    return y_true, y_pred

for name, cm in [("Stage 1", cm1), ("Stage 2", cm2)]:
    yt, yp = expand(cm)
    print(name, "acc", round(accuracy_score(yt, yp), 4),
          "macro", round(f1_score(yt, yp, average="macro"), 4),
          "weighted", round(f1_score(yt, yp, average="weighted"), 4),
          "micro", round(f1_score(yt, yp, average="micro"), 4))

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
print("nltk BLEU-4:", round(sentence_bleu([ref], cand), 4))
sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"]).score(" ".join(ref), " ".join(cand))
print({k: round(v.fmeasure, 4) for k, v in sc.items()})

Stage 1 acc 0.7667 macro 0.7546 weighted 0.7659 micro 0.7667
Stage 2 acc 0.8467 macro 0.8406 weighted 0.8472 micro 0.8467


nltk BLEU-4: 0.475
{'rouge1': 0.8421, 'rouge2': 0.5882, 'rougeL': 0.5263}


In [15]:
# collect the numbers used in the report
import json
out = dict(
    acc1=acc1, se1=se1, wald1=wald1, wil1=wil1, acc2=acc2, se2=se2, wald2=wald2, wil2=wil2,
    pc1=pc1.round(4).to_dict(), pc2=pc2.round(4).to_dict(), m1=m1, m2=m2,
    mean=mean, std=std_sample, std_pop=std_pop, precisions=precisions, bp=bp, bleu4=bleu4,
    rouge=rouge, lcs=lcs, meteor=dict(m=m, P=P, R=R, fmean=fmean, chunks=chunks, penalty=penalty, score=meteor),
    nll=nll, avg_nll=avg_nll, ppl=ppl,
)
json.dump(out, open("q4_results.json", "w"), indent=1, default=float)
print("saved")

saved
